<a href="https://colab.research.google.com/github/Uczas/TechCrush-AI-ML-Course/blob/main/Group_Juliet_Real_Time_Object_Detection_For_The_Visually_Impaired.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Group Juliet")
print("Real-time Object Detection For The Visually Impaired")

In [ ]:
print(" --- installing the required libraries ---")
!pip install pyttsx3 ultralytics opencv-python numpy

In [ ]:
#The pyttsx3 library requires an external speech engine like eSpeak or eSpeak-ng.
#This cell will install espeak-ng.

!apt-get install espeak-ng

In [ ]:
import cv2  # OpenCV for camera capture and image processing
import numpy as np  # For numerical operations and area calculations
import pyttsx3  # Text-to-speech library for audio feedback
import time  # For timing and cooldown management
from ultralytics import YOLO  # YOLO model for object detection
import threading  # For running TTS without blocking the main loop
from collections import defaultdict  # For tracking last alert times per object
import uuid # For generating unique filenames for audio output in Colab
from IPython.display import Audio, display # For playing audio in Colab output

# ============================================================================
# COLAB SPECIFIC IMPORTS AND UTILITIES
# ============================================================================

# These are required to use the webcam and display frames in Google Colab
from google.colab.output import eval_js
from base64 import b64decode, b64encode


def video_frame_generator():
    """
    Generates video frames from the webcam in Google Colab.
    This function is adapted from a common Colab pattern for webcam access.
    """
    js = Javascript('''
        var video;
        var canvas;
        var ctx;

        async function setupCamera() {
            if (video) return; // Already set up

            const div = document.createElement('div');
            document.body.appendChild(div);
            video = document.createElement('video');
            video.style = 'display:none;'; // Hide video element
            div.appendChild(video);

            await navigator.mediaDevices.getUserMedia({video: true}).then(function(stream) {
                video.srcObject = stream;
                video.play();
            });

            await new Promise((resolve) => video.onplaying = resolve);

            canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            // div.appendChild(canvas); // Don't append canvas to DOM if not directly displayed
            ctx = canvas.getContext('2d');
        }

        async function getFrame() {
            await setupCamera(); // Ensure camera is set up
            ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
            return canvas.toDataURL('image/jpeg', 0.8);
        }
    ''')

    display(js) # This runs the JS to define setupCamera and getFrame

    # Now, repeatedly call getFrame from Python
    while True:
        frame_data = eval_js('getFrame()')
        yield frame_data


def b64_to_ndarray(b64_string):
    """
    Converts a base64 encoded JPEG image string to a numpy array (OpenC V format).
    """
    img_bytes = b64decode(b64_string.split(',')[1])
    nparr = np.frombuffer(img_bytes, np.uint8)
    img_np = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return img_np

# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Detection parameters
CONFIDENCE_THRESHOLD = 0.5  # Minimum confidence score (0-1) to consider a detection
FRAME_SKIP = 2  # Process every Nth frame to reduce lag (higher = faster but less accurate)

# Proximity estimation parameters (based on bounding box area percentage)
# Larger bounding box = object is closer
CLOSE_THRESHOLD_PERCENT = 15  # Area > 15% of frame = "close"
MEDIUM_THRESHOLD_PERCENT = 5  # Area > 5% but < 15% = "medium distance"

# Field of view parameters (only objects straight ahead)
CENTER_ZONE_WIDTH_PERCENT = 40  # Only consider objects in central 40% of frame width

# Audio feedback parameters
COOLDOWN_SECONDS = 3  # Don't repeat same object type within 3 seconds
MAX_OBJECTS_PER_ANNOUNCEMENT = 3  # Max objects to announce at once (avoid overload)

# Camera settings
CAMERA_ID = 0  # 0 for built-in webcam, 1 for external USB camera
FRAME_WIDTH = 640  # Reduce resolution for better performance
FRAME_HEIGHT = 480

# Important obstacles to detect (filtered from COCO dataset)
# These are the most relevant objects for navigation safety
IMPORTANT_OBJECTS = {
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'dog', 'cat',
    'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet',
    'tv', 'laptop', 'keyboard', 'cell phone', 'oven', 'refrigerator',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase',
    'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat',
    'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog',
    'pizza', 'donut', 'cake'
}

# ============================================================================
# AUDIO MANAGER CLASS
# ============================================================================

class AudioFeedbackManager:
    """
    Manages text-to-speech feedback with cooldown mechanisms to prevent
    overwhelming the user with constant audio callouts.
    """

    def __init__(self):
        """
        Initialize the TTS engine and tracking variables.
        """
        self.tts_engine = pyttsx3.init()
        self.last_alert_time = defaultdict(float)  # Tracks last time each object type was announced
        self.speech_queue = []  # Queue for pending speech (FIFO)
        self.is_speaking = False
        self.lock = threading.Lock()  # Thread safety for queue access
        self.audio_display_handle = None # For updating Colab audio output

        # Configure TTS voice properties for better clarity
        self._configure_voice()

        # Start background thread for speech processing
        self.speech_thread = threading.Thread(target=self._process_speech_queue, daemon=True)
        self.speech_thread.start()

    def _configure_voice(self):
        """Configure TTS voice speed and volume for optimal listening."""
        # Get available voices
        voices = self.tts_engine.getProperty('voices')

        # Try to use a clear, understandable voice (prefer English female if available)
        for voice in voices:
            if 'english' in voice.name.lower() or 'en' in voice.id.lower():
                self.tts_engine.setProperty('voice', voice.id)
                break

        # Set speech rate (words per minute) - slower is clearer
        self.tts_engine.setProperty('rate', 150)  # Default is ~200

        # Set volume (0.0 to 1.0)
        self.tts_engine.setProperty('volume', 0.9)

    def can_announce(self, object_name, current_time):
        """
        Check if an object can be announced based on cooldown.

        Args:
            object_name: The type of object detected
            current_time: Current timestamp in seconds

        Returns:
            bool: True if the object can be announced, False otherwise
        """
        last_time = self.last_alert_time[object_name]
        return (current_time - last_time) >= COOLDOWN_SECONDS

    def update_alert_time(self, object_name, current_time):
        """
        Update the last alert time for an object.
        """
        self.last_alert_time[object_name] = current_time

    def queue_speech(self, message):
        """
        Queue a speech message for background processing.

        Args:
            message: String message to speak
        """
        with self.lock:
            self.speech_queue.append(message)

    def _process_speech_queue(self):
        """
        Background thread that processes queued speech messages.
        This prevents TTS from blocking the main detection loop.
        """
        while True:
            if self.speech_queue and not self.is_speaking:
                with self.lock:
                    message = self.speech_queue.pop(0);

                self.is_speaking = True

                # --- Colab specific audio activation ---
                # pyttsx3 does not directly output audio in Colab's server environment.
                # To activate audible feedback, we generate a temporary audio file
                # and then play it using IPython.display.Audio.
                # This will create an audio player in the notebook output for each announcement.
                # Be aware that this can generate many audio players and may be disruptive
                # or cause delays due to file I/O and browser rendering.
                temp_audio_file = f"colab_tts_{uuid.uuid4()}.wav"
                self.tts_engine.save_to_file(message, temp_audio_file)
                self.tts_engine.runAndWait() # This blocks the speech thread until audio is saved

                # Display the audio in Colab output, updating the same display area
                audio_to_display = Audio(temp_audio_file, autoplay=True)
                if self.audio_display_handle is None:
                    self.audio_display_handle = display(audio_to_display, display_id='colab_audio_output')
                else:
                    self.audio_display_handle.update(audio_to_display)

                # It's recommended to clean up temporary audio files if many are generated.
                # Uncomment the following lines if you want to automatically remove them:
                # import os
                # os.remove(temp_audio_file)
                # --- End Colab specific audio activation ---

                self.is_speaking = False

            time.sleep(0.05)

    def announce_obstacles(self, obstacles):
        """
        Announce a list of obstacles with appropriate language.

        Args:
            obstacles: List of (object_name, proximity_description, area_percent) tuples
        """
        if not obstacles:
            return

        current_time = time.time()
        filtered_obstacles = []

        # Apply cooldown filtering
        for obj_name, proximity, area in obstacles:
            if self.can_announce(obj_name, current_time):
                filtered_obstacles.append((obj_name, proximity, area))
                self.update_alert_time(obj_name, current_time)

        # Limit the number of objects announced at once
        filtered_obstacles = filtered_obstacles[:MAX_OBJECTS_PER_ANNOUNCEMENT]

        if not filtered_obstacles:
            return

        # Construct the announcement message
        if len(filtered_obstacles) == 1:
            obj_name, proximity, _ = filtered_obstacles[0]
            message = f"{proximity} {obj_name}"
        else:
            # Multiple objects: "close person, medium chair"
            parts = [f"{proximity} {obj_name}" for obj_name, proximity, _ in filtered_obstacles]
            if len(parts) == 2:
                message = f"{parts[0]} and {parts[1]}"
            else:
                message = ", ".join(parts[:-1]) + f", and {parts[-1]}"

        # Queue the message for speaking
        self.queue_speech(message)


# ============================================================================
# OBJECT DETECTION MANAGER CLASS
# ============================================================================

class ObjectDetectionSystem:
    """
    Main system that handles camera capture, object detection, and
    spatial analysis for proximity estimation.
    """

    def __init__(self, model_path='yolov8n.pt'):
        """
        Initialize the detection system.

        Args:
            model_path: Path to YOLO model weights (auto-downloads if not exists)
        """
        # Load YOLO model (ultralytics will download automatically if not found)
        print(f"[INFO] Loading YOLO model from {model_path}...")
        self.model = YOLO(model_path)
        print("[INFO] Model loaded successfully!")

        # Initialize audio manager
        self.audio_manager = AudioFeedbackManager()

        # Frame counter for skipping frames (reduces lag)
        self.frame_count = 0

        # Camera capture object (will be a generator for Colab)
        self.webcam_generator = None

        # Frame dimensions
        self.frame_width = FRAME_WIDTH
        self.frame_height = FRAME_HEIGHT

        # Speech toggle flag
        self.speech_enabled = True

        # For Colab display
        self.display_handle = None

    def setup_camera(self):
        """
        Initialize and configure the camera for Colab.
        """
        print("[INFO] Setting up webcam for Colab...")
        self.webcam_generator = video_frame_generator()
        print(f"[INFO] Webcam initialized for Colab.")
        return True

    def estimate_proximity(self, bbox_area, total_area):
        """
        Estimate object proximity based on bounding box area percentage.

        Args:
            bbox_area: Area of the detected object's bounding box
            total_area: Total frame area

        Returns:
            tuple: (proximity_text, area_percentage)
        """
        area_percentage = (bbox_area / total_area) * 100

        if area_percentage >= CLOSE_THRESHOLD_PERCENT:
            proximity = "close"
        elif area_percentage >= MEDIUM_THRESHOLD_PERCENT:
            proximity = "medium"
        else:
            proximity = "far"

        return proximity, area_percentage

    def is_in_center_zone(self, bbox_center_x, frame_width):
        """
        Check if an object is in the center zone (directly ahead).

        Args:
            bbox_center_x: X-coordinate of the bounding box center
            frame_width: Total frame width

        Returns:
            bool: True if object is in center zone, False otherwise
        """
        center_threshold = (frame_width * CENTER_ZONE_WIDTH_PERCENT) / 200
        frame_center = frame_width / 2

        return abs(bbox_center_x - frame_center) <= center_threshold

    def get_relative_position(self, bbox_center_x, frame_width):
        """
        Get the relative position description (left/center/right).

        Args:
            bbox_center_x: X-coordinate of the bounding box center
            frame_width: Total frame width

        Returns:
            str: Position description
        """
        frame_center = frame_width / 2
        left_threshold = frame_center - (frame_width * 0.2)
        right_threshold = frame_center + (frame_width * 0.2)

        if bbox_center_x < left_threshold:
            return "left"
        elif bbox_center_x > right_threshold:
            return "right"
        else:
            return "ahead"

    def process_frame(self, frame):
        """
        Process a single frame for object detection and audio feedback.

        Args:
            frame: Input image frame from camera

        Returns:
            numpy.ndarray: Annotated frame for visualization
        """
        total_area = frame.shape[0] * frame.shape[1]
        detected_obstacles = []  # List of (name, proximity, area, position)

        # Run YOLO inference
        # Note: Running on the full frame - no modifications to the pre-trained model
        results = self.model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)

        # Extract detection results
        for result in results:
            boxes = result.boxes

            if boxes is not None:
                for box in boxes:
                    # Get bounding box coordinates
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    bbox_area = (x2 - x1) * (y2 - y1)

                    # Get class name and confidence
                    class_id = int(box.cls[0])
                    class_name = self.model.names[class_id]
                    confidence = float(box.conf[0])

                    # Filter only important obstacles
                    if class_name not in IMPORTANT_OBJECTS:
                        continue

                    # Calculate bounding box center
                    bbox_center_x = (x1 + x2) / 2

                    # Check if object is in center zone (directly ahead)
                    if not self.is_in_center_zone(bbox_center_x, frame.shape[1]):
                        continue

                    # Estimate proximity
                    proximity, area_percent = self.estimate_proximity(bbox_area, total_area)

                    # Only announce objects that are close or medium (ignore far objects)
                    if proximity in ['close', 'medium']:
                        position = self.get_relative_position(bbox_center_x, frame.shape[1])
                        detected_obstacles.append((class_name, proximity, area_percent, position, confidence))

                        # Draw detection on frame (for visual feedback/debugging)
                        self._draw_detection(frame, x1, y1, x2, y2, class_name,
                                            confidence, proximity, position)

        # Announce detected obstacles (only if speech is enabled)
        if detected_obstacles and self.speech_enabled:
            # Format obstacles for audio (combine proximity and position)
            audio_obstacles = [(name, prox, area) for name, prox, area, pos, conf in detected_obstacles]
            self.audio_manager.announce_obstacles(audio_obstacles)

        # Add status text to frame
        frame = self._draw_status(frame, len(detected_obstacles))

        return frame

    def _draw_detection(self, frame, x1, y1, x2, y2, class_name, confidence, proximity, position):
        """
        Draw bounding box and labels on the frame for visualization.

        Args:
            frame: Image frame to draw on
            x1, y1, x2, y2: Bounding box coordinates
            class_name: Detected object name
            confidence: Detection confidence score
            proximity: Proximity description (close/medium/far)
            position: Position description (left/center/right)
        """
        # Color mapping based on proximity (close=red, medium=orange, far=green)
        colors = {
            'close': (0, 0, 255),      # Red (BGR format)
            'medium': (0, 165, 255),   # Orange
            'far': (0, 255, 0)         # Green
        }
        color = colors.get(proximity, (255, 255, 255))

        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # Create label text
        label = f"{class_name}: {confidence:.2f} ({proximity}, {position})"

        # Draw label background
        (label_width, label_height), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2
        )
        cv2.rectangle(frame, (x1, y1 - label_height - 10),
                     (x1 + label_width, y1), color, -1)

        # Draw label text
        cv2.putText(frame, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    def _draw_status(self, frame, num_obstacles):
        """
        Draw status information on the frame.

        Args:
            frame: Image frame to draw on
            num_obstacles: Number of detected obstacles in current frame

        Returns:
            numpy.ndarray: Frame with status overlay
        """
        # Semi-transparent overlay for status bar
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (frame.shape[1], 40), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

        # Speech status indicator
        speech_status = "ON" if self.speech_enabled else "OFF"
        speech_color = (0, 255, 0) if self.speech_enabled else (0, 0, 255)

        # Status text
        status_text = f"Obstacles ahead: {num_obstacles} | Cooldown: {COOLDOWN_SECONDS}s | Speech: {speech_status}"
        cv2.putText(frame, status_text, (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.55, speech_color, 2)

        # Instruction text
        cv2.putText(frame, "Press 'q' to quit | 's' to toggle speech",
                   (10, frame.shape[0] - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        return frame

    def run(self):
        """
        Main loop for the object detection system.
        """
        if not self.setup_camera():
            return

        print("\n[INFO] System starting...")
        print("[INFO] Audio feedback will announce close/medium obstacles directly ahead")
        print("[INFO] Press 'q' to quit | 's' to toggle speech feedback")
        print("[INFO] Ready!\n")

        try:
            # Use a generator to get frames from Colab's webcam utility
            for b64_frame in self.webcam_generator:
                # Convert base64 string to numpy array (OpenCV image)
                frame = b64_to_ndarray(b64_frame)

                # Resize frame for consistent processing
                frame = cv2.resize(frame, (self.frame_width, self.frame_height))

                # Skip frames to reduce lag (process every Nth frame)
                self.frame_count += 1
                if self.frame_count % FRAME_SKIP != 0:
                    # Encode frame to JPEG for display in Colab
                    _, encimg = cv2.imencode('.jpg', frame)
                    if self.display_handle is None:
                        self.display_handle = display(Image(data=encimg.tobytes()), display_id=True)
                    else:
                        self.display_handle.update(Image(data=encimg.tobytes()))
                    continue

                # Process frame for detection
                processed_frame = self.process_frame(frame)

                # Encode processed frame to JPEG for display in Colab
                _, encimg = cv2.imencode('.jpg', processed_frame)
                if self.display_handle is None:
                    self.display_handle = display(Image(data=encimg.tobytes()), display_id=True)
                else:
                    self.display_handle.update(Image(data=encimg.tobytes()))

                # Colab doesn't have direct key press handling like cv2.waitKey
                # User needs to interrupt execution to stop.
                # Toggling speech would require a dedicated Colab widget or input,
                # which is beyond a simple code fix here.
                # For now, 's' and 'q' functionality from cv2.waitKey will not work.

        except KeyboardInterrupt:
            print("\n[INFO] System interrupted by user")
        except Exception as e:
            print(f"[ERROR] An error occurred: {e}")
        finally:
            self.cleanup()

    def cleanup(self):
        """
        Clean up resources (camera, windows).
        """
        print("[INFO] Shutting down system...")
        # In Colab, there's no cv2.destroyAllWindows() or cap.release() in this context
        # The JavaScript stream will eventually be garbage collected.
        print("[INFO] Cleanup complete. Goodbye!")


# ============================================================================
# MAIN ENTRY POINT
# ============================================================================

def main():
    """
    Main function to start the object detection system.
    """
    print("=" * 60)
    print("REAL-TIME OBJECT DETECTION FOR VISUALLY IMPAIRED USERS")
    print("=" * 60)
    print("\nThis system provides audio feedback about obstacles in your path.")
    print("Only objects that are close/medium and directly ahead are announced.")
    print("This prevents overwhelming the user with constant audio callouts.\n")

    # Check if required packages are installed
    try:
        import ultralytics
        import cv2
        import pyttsx3
    except ImportError as e:
        print(f"[ERROR] Missing required package: {e}")
        print("\nPlease install required packages using:")
        print("pip install ultralytics opencv-python pyttsx3 numpy")
        return

    # Create and run the detection system
    detection_system = ObjectDetectionSystem()
    detection_system.run()


if __name__ == "__main__":
    main()

In [ ]:
# To see the .wav files generated by the system,
# you can list the files in the current directory.
# Run this cell

import glob

# List all .wav files in the current directory that start with 'colab_tts_'
audio_files = glob.glob('colab_tts_*.wav')

if audio_files:
    print('Generated audio files:')
    for f in audio_files:
        print(f)
else:
    print('No audio files found starting with colab_tts_')